# ULTRON — Google Colab Free Runtime Diagnostic

Run this notebook manually inside the Colab interface to inspect the runtime Google actually assigned. It reports CPU, system RAM, disk, Python, PyTorch and GPU availability.

**Safety:** this notebook does not train ULTRON, change model weights, access Supabase, read/write checkpoints, mount Google Drive, or call an external AI API. Colab Free resources are dynamic and not guaranteed. Managed free runtimes must be operated through the notebook UI; do not connect them to remote dashboard control.

In [ ]:
import json, os, platform, shutil, subprocess, sys
from pathlib import Path

def read_meminfo():
    values = {}
    path = Path('/proc/meminfo')
    if path.exists():
        for line in path.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 2 and parts[1].isdigit():
                values[parts[0].rstrip(':')] = int(parts[1]) * 1024
    return values

memory = read_meminfo()
disk = shutil.disk_usage('/')
report = {
    'provider': 'google_colab_manual',
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'logical_cpus': os.cpu_count(),
    'memory_total_bytes': memory.get('MemTotal'),
    'memory_available_bytes': memory.get('MemAvailable'),
    'root_disk_total_bytes': disk.total,
    'root_disk_free_bytes': disk.free,
    'training_started': False,
    'model_weights_modified': False,
    'persistent_checkpoints_touched': False,
}
try:
    import torch
    report['pytorch_version'] = torch.__version__
    report['pytorch_cpu_threads'] = torch.get_num_threads()
    report['cuda_available'] = torch.cuda.is_available()
    if torch.cuda.is_available():
        report['gpu_name'] = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        report['gpu_memory_bytes'] = props.total_memory
    else:
        report['gpu_name'] = None
        report['gpu_memory_bytes'] = None
except Exception as exc:
    report['pytorch_error'] = type(exc).__name__ + ': ' + str(exc)

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=10)
    report['nvidia_smi'] = result.stdout.strip() if result.returncode == 0 else 'No NVIDIA GPU reported.'
except (FileNotFoundError, subprocess.TimeoutExpired):
    report['nvidia_smi'] = 'nvidia-smi unavailable; no NVIDIA GPU confirmed.'

print(json.dumps(report, indent=2))

## How to interpret the result

- `cuda_available: true` means this session currently has a usable CUDA GPU. Record its model and memory; do not assume the next session receives the same GPU.
- `cuda_available: false` means this session is CPU-only. You can check **Runtime → Change runtime type** and select a GPU if one is offered.
- Colab Free limits and GPU availability change dynamically. Google documents a maximum runtime of up to 12 hours depending on availability and usage patterns, not a guaranteed 12-hour session.
- Keep this provider manual-only. Do not use SSH, remote desktop, or a separate web dashboard to control a managed free Colab runtime.

Official limits: https://research.google.com/colaboratory/faq.html